In [3]:
import pandas as pd
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

df = pd.read_csv('IndoLeX_Database.csv')
text_column = "word"

def case_folding(text):
    return str(text).lower()
text = "Saya Belajar NLP"
print(case_folding(text))

def remove_url(text):
    return re.sub(r"http\S+|www\S+", "", text)
text = "Kunjungi https://example.com sekarang"
print(remove_url(text))

def remove_mention(text):
    return re.sub(r"@\w+", "", text)
text = "@user1 produk ini bagus"
print(remove_mention(text))

def remove_hashtag(text):
    return re.sub(r"#\w+", "", text)

def remove_number(text):
    return re.sub(r"\d+", "", text)

def remove_punctuation(text):
    return text.translate(
        str.maketrans("", "", string.punctuation)
    )
def clean_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

def clean_text(text):
    text = str(text)
    text = text.lower()           # case folding
    text = remove_url(text)       # hapus URL
    text = remove_mention(text)   # hapus mention
    text = remove_hashtag(text)   # hapus hashtag
    text = remove_number(text)    # hapus angka
    text = remove_punctuation(text)  # hapus punctuation
    text = clean_whitespace(text) # bersihkan whitespace
    return text
df["clean_text"] = df[text_column].apply(clean_text)
df[[text_column, "clean_text"]].head(10)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


saya belajar nlp
Kunjungi  sekarang
 produk ini bagus


,word,clean_text
0,aba,aba
1,peraba,peraba
2,perabaan,perabaan
3,teraba,teraba
4,beraba,beraba
5,seaba,seaba
6,pengaba,pengaba
7,abakan,abakan
8,mengaba,mengaba
9,abamu,abamu


In [ ]:
stop_words = set(stopwords.words("indonesian"))
def remove_stopwords(tokens):
    return [
        token for token in tokens
        if token not in stop_words
    ]
tokens = ["saya", "sedang", "belajar", "nlp", "yang", "menarik"]
print(remove_stopwords(tokens))

factory = StemmerFactory()
stemmer = factory.create_stemmer()
def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]
text = "Mahasiswa sedang mempelajari pemrograman."
print(stemmer.stem(text))

def preprocessing_pipeline(text):
    # 1. Cleaning
    text = clean_text(text)
    # 2. Tokenization
    tokens = word_tokenize(text)
    # 3. Stopword removal
    tokens = remove_stopwords(tokens)
    # 4. Stemming
    tokens = stem_tokens(tokens)
    return tokens
# Menerapkan ke dataset
processed_tokens = []
total_rows = len(df)
for index, text in enumerate(df[text_column], start=1):
    processed_tokens.append(preprocessing_pipeline(text))
    if index % 10000 == 0 or index == total_rows:
        print(f"Selesai preprocessing {index:,}/{total_rows:,} baris")
df["processed_tokens"] = processed_tokens

df["processed_text"] = df["processed_tokens"].apply(
    lambda tokens: " ".join(tokens)
)

for i in range(5):
    print("RAW:")
    print(df.iloc[i][text_column])
    print("\nPROCESSED:")
    print(df.iloc[i]["processed_text"])
    print("=" * 80)

output_columns = [text_column, "clean_text", "processed_text"]
output_path = "IndoLeX_Database_preprocessed.csv"
temporary_path = output_path + ".tmp"
df[output_columns].to_csv(
    temporary_path,
    index=False
)
import os
os.replace(temporary_path, output_path)
print("Dataset berhasil disimpan.")

['belajar', 'nlp', 'menarik']
mahasiswa sedang ajar pemrograman
